# Data Cleaning Exercises
Practical Python exercises based on real interview problems. Each exercise targets a specific pattern you need to own cold.

**How to use:** Read the problem, fill in the `# YOUR CODE HERE` sections, run the cell. If the assert passes silently, you got it right.

---
## Exercise 1 — Dict access and iteration

The most basic thing. You need to be able to access fields and iterate over key/value pairs without thinking.

In [2]:
record = {"id": "x1", "name": "trade", "amount": 500, "currency": "USD"}

# 1a. Access the 'amount' field
amount = record["amount"]
assert amount == 500

# 1b. Access a field that might not exist, default to 0 if missing
fee = record.get("fee", 0)
assert fee == 0

# 1c. Collect all keys where the value is a string
string_keys = []
for key, value in record.items():
    if isinstance(value, str):
        string_keys.append(key)

assert sorted(string_keys) == ["currency", "id", "name"]

print("Exercise 1 passed")

Exercise 1 passed


---
## Exercise 2 — Finding duplicates with a set

Given a list of records, find all IDs that appear more than once.

In [7]:
records = [
    {"id": "a1", "value": 10},
    {"id": "b2", "value": 20},
    {"id": "a1", "value": 99},  # duplicate
    {"id": "c3", "value": 30},
    {"id": "b2", "value": 55},  # duplicate
]

# Return a list of IDs that appear more than once
def find_duplicates(records):
    seen = set()
    duplicates = set()
    for record in records:
        id = record["id"]

        if id in seen:
            duplicates.add(id)
        else: 
            seen.add(id)

    return list(duplicates)

assert sorted(find_duplicates(records)) == ["a1", "b2"]
print("Exercise 2 passed")

Exercise 2 passed


---
## Exercise 3 — Safe type conversion

Real data is messy. Values come in as strings with commas, currency symbols, or are None entirely. Convert them safely.

In [12]:
# 3a. Convert each raw value to a float. If it can't be converted, store None.
raw_values = ["1,234.50", "99.9", "£500", "abc", None, "2.0"]

def clean_value(v):
    if v is None:
        return None
    try:
        return float(str(v).replace(",", ""))
    except ValueError:
        return None


    # strip commas and currency symbols (hint: .replace() chained, or strip non-numeric chars)
    # try converting to float, return None if it fails
    # YOUR CODE HERE
    pass

results = [clean_value(v) for v in raw_values]
assert results == [1234.50, 99.9, None, None, None, 2.0]
# note: £500 returns None because £ can't be stripped cleanly without knowing all currency symbols
# acceptable to handle that as invalid

print("Exercise 3 passed")

Exercise 3 passed


---
## Exercise 4 — The continue pattern

This is the core pattern for validation loops. For each record, validate it. If anything is wrong, append to errors and continue. Only add to clean if everything passes.

In [14]:
trades = [
    {"id": "t1", "amount": "500",  "currency": "USD"},
    {"id": "t2", "amount": None,   "currency": "GBP"},   # null amount
    {"id": "t3", "amount": "abc",  "currency": "EUR"},   # invalid amount
    {"id": "t1", "amount": "200",  "currency": "USD"},   # duplicate id
    {"id": "t4", "amount": "1,000","currency": "JPY"},
]

def validate_trades(trades):
    clean = []
    errors = []
    seen_ids = set()
    
    for trade in trades:
        trade_id = trade["id"]
        amount = trade["amount"]
        
        # check duplicate
        if trade_id in seen_ids:
            errors.append({"id": trade_id, "reason": "duplicate", "raw": trade})
            continue
        # check null
        if amount == None:
            errors.append({"id": trade_id, "reason": "invalid amount","raw": trade})
            seen_ids.add(trade_id)
            continue

        # convert amount to float (strip commas)
        try:
            numeric_amount = float(str(amount).replace(",", ""))
        except (ValueError, TypeError):
            errors.append({"id": trade_id, "reason": "invalid amount", "raw": trade})
            seen_ids.add(trade_id)
            continue
        
        # if we get here, everything is valid
        seen_ids.add(trade_id)
        clean.append({"id": trade_id, "amount": numeric_amount, "currency": trade["currency"]})  # replace None with numeric_amount
    
    return clean, errors

clean, errors = validate_trades(trades)
assert len(clean) == 2, f"Expected 2 clean trades, got {len(clean)}"
assert len(errors) == 3, f"Expected 3 errors, got {len(errors)}"
assert clean[0]["amount"] == 500.0
assert clean[1]["amount"] == 1000.0
print("Exercise 4 passed")

Exercise 4 passed


---
## Exercise 5 — Date parsing

Dates come in every format imaginable. Parse them all to a consistent datetime, or flag as invalid.

In [ ]:
from dateutil import parser as dateparser
import pandas as pd

raw_dates = [
    "2026-05-01 10:00",
    "01/05/2026 10:00:00",
    "2026-05-02T12:00:00Z",   # timezone-aware
    "bad-date",
    "May 3 2026",
]

# 5a. Parse each date using dateutil. Strip timezone info. Return None for invalid.
def parse_date(raw):
    
    pass

results = [parse_date(d) for d in raw_dates]
assert results[3] is None, "bad-date should return None"
assert all(r.tzinfo is None for r in results if r is not None), "all datetimes should be timezone-naive"
print("5a passed")

# 5b. Do the same with pandas
def parse_date_pandas(raw):
    # YOUR CODE HERE
    pass

results_pd = [parse_date_pandas(d) for d in raw_dates]
assert results_pd[3] is None
print("5b passed")

---
## Exercise 6 — Full problem (interview style)

This is the same shape as the BofA interview question. Clean a list of market events: normalise dates, clean values, handle duplicates and nulls. Return `(clean_events, errors)` sorted by `event_time`.

In [ ]:
from dateutil import parser as dateparser

market_data = [
    {"event_id": "e1", "event_time": "2026-05-01 09:00", "price": "102.5"},
    {"event_id": "e2", "event_time": "01/05/2026 09:30", "price": "1,103.0"},
    {"event_id": "e3", "event_time": "2026-05-01T10:00:00Z", "price": "101.0"},
    {"event_id": "e2", "event_time": "2026-05-01 09:30", "price": "103.5"},  # duplicate
    {"event_id": "e4", "event_time": "not-a-date", "price": "99.0"},         # bad date
    {"event_id": "e5", "event_time": "2026-05-01 11:00", "price": None},      # null price
    {"event_id": "e6", "event_time": "2026-05-01 08:00", "price": "100.0"},
]

def clean_market_data(events):
    # YOUR CODE HERE
    pass

clean, errors = clean_market_data(market_data)

assert len(clean) == 4, f"Expected 4 clean events, got {len(clean)}"
assert len(errors) == 3, f"Expected 3 errors, got {len(errors)}"
# should be sorted by event_time — e6 (08:00) comes first
assert clean[0]["event_id"] == "e6", "should be sorted by event_time"
assert clean[1]["price"] == 102.5
print("Exercise 6 passed — full problem solved")

---
## Exercise 7 — Grouping and aggregation

A common follow-up: once you have clean data, group it and compute something. Group events by date and find the average price per day.

In [ ]:
from collections import defaultdict

events = [
    {"event_id": "e1", "date": "2026-05-01", "price": 100.0},
    {"event_id": "e2", "date": "2026-05-01", "price": 102.0},
    {"event_id": "e3", "date": "2026-05-02", "price": 98.0},
    {"event_id": "e4", "date": "2026-05-02", "price": 104.0},
    {"event_id": "e5", "date": "2026-05-02", "price": 99.0},
]

# Return a dict of {date: average_price} rounded to 2 decimal places
def average_price_by_date(events):
    # hint: use defaultdict(list) to group prices by date
    # YOUR CODE HERE
    pass

result = average_price_by_date(events)
assert result == {"2026-05-01": 101.0, "2026-05-02": 100.33}
print("Exercise 7 passed")

---
## Solutions
Only look at these after genuinely attempting each exercise.

In [ ]:
# --- Exercise 1 ---
amount = record["amount"]
fee = record.get("fee", 0)
# for key, value in record.items():

# --- Exercise 2 ---
# for record in records:
#     rid = record["id"]
#     if rid in seen:
#         duplicates.add(rid)
#     seen.add(rid)

# --- Exercise 3 ---
# def clean_value(v):
#     if v is None:
#         return None
#     try:
#         return float(str(v).replace(",", ""))
#     except ValueError:
#         return None

# --- Exercise 4 ---
# if trade_id in seen_ids:
#     errors.append({"id": trade_id, "reason": "duplicate", "raw": trade})
#     continue
# if amount is None:
#     errors.append({"id": trade_id, "reason": "null amount", "raw": trade})
#     seen_ids.add(trade_id)
#     continue
# try:
#     numeric_amount = float(str(amount).replace(",", ""))
# except ValueError:
#     errors.append({"id": trade_id, "reason": "invalid amount", "raw": trade})
#     seen_ids.add(trade_id)
#     continue

# --- Exercise 5a ---
# def parse_date(raw):
#     try:
#         return dateparser.parse(raw).replace(tzinfo=None)
#     except Exception:
#         return None

# --- Exercise 5b ---
# def parse_date_pandas(raw):
#     try:
#         return pd.to_datetime(raw).to_pydatetime().replace(tzinfo=None)
#     except Exception:
#         return None

# --- Exercise 6 ---
# def clean_market_data(events):
#     clean_events = []
#     errors = []
#     seen_ids = set()
#     for event in events:
#         event_id = event["event_id"]
#         raw_time = event["event_time"]
#         price = event["price"]
#         if event_id in seen_ids:
#             errors.append({"event_id": event_id, "reason": "duplicate", "raw": event})
#             continue
#         if price is None:
#             errors.append({"event_id": event_id, "reason": "null price", "raw": event})
#             seen_ids.add(event_id)
#             continue
#         try:
#             numeric_price = float(str(price).replace(",", ""))
#         except ValueError:
#             errors.append({"event_id": event_id, "reason": "invalid price", "raw": event})
#             seen_ids.add(event_id)
#             continue
#         try:
#             parsed_time = dateparser.parse(raw_time).replace(tzinfo=None)
#         except Exception:
#             errors.append({"event_id": event_id, "reason": "invalid date", "raw": event})
#             seen_ids.add(event_id)
#             continue
#         seen_ids.add(event_id)
#         clean_events.append({"event_id": event_id, "event_time": parsed_time, "price": numeric_price})
#     clean_events.sort(key=lambda x: x["event_time"])
#     return clean_events, errors

# --- Exercise 7 ---
# def average_price_by_date(events):
#     groups = defaultdict(list)
#     for event in events:
#         groups[event["date"]].append(event["price"])
#     return {date: round(sum(prices) / len(prices), 2) for date, prices in groups.items()}

print("Solutions loaded — but you should have tried first")